<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Air-Quality-_PM-2.5/PM_2_5_Without_iteration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Input

# =========================
# 1. Load datasets
# =========================
train_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/PM 2.5/pm25_training_dataset.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/PM 2.5/pm25_testing_dataset.csv")

features = [
    "air_quality_PM10",
    "air_quality_Carbon_Monoxide",
    "air_quality_Nitrogen_dioxide",
    "air_quality_Sulphur_dioxide",
    "humidity",
    "cloud",
    "visibility_km",
    "longitude",
    "temperature_celsius",
    "condition_text",
    "uv_index",
    "wind_mph",
    "precip_mm",
    "gust_mph",
    "wind_degree"
]

target = "air_quality_PM2.5"

# =========================
# 2. Keep needed columns
# =========================
train_df = train_df[features + [target]].copy()
test_df = test_df[features + [target]].copy()

# condition_text already encoded
train_df["condition_text"] = pd.to_numeric(train_df["condition_text"], errors="coerce")
test_df["condition_text"] = pd.to_numeric(test_df["condition_text"], errors="coerce")

# convert all columns to numeric if needed
for col in features + [target]:
    train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
    test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

# drop missing values
train_df = train_df.dropna()
test_df = test_df.dropna()

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

# =========================
# 3. Evaluation function
# =========================
def evaluate_model(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true_safe = np.where(np.array(y_true) == 0, 1e-10, y_true)
    accuracy = 100 - (np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100)

    return mse, rmse, mae, r2, accuracy

# Linear (SGD)

In [ ]:
scaler_sgd = StandardScaler()
X_train_sgd = scaler_sgd.fit_transform(X_train)
X_test_sgd = scaler_sgd.transform(X_test)

sgd_model = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
sgd_model.fit(X_train_sgd, y_train)

y_train_pred_sgd = sgd_model.predict(X_train_sgd)
y_test_pred_sgd = sgd_model.predict(X_test_sgd)

train_metrics_sgd = evaluate_model(y_train, y_train_pred_sgd)
test_metrics_sgd = evaluate_model(y_test, y_test_pred_sgd)

print("Linear (SGD) - Training Results")
print("MSE:", train_metrics_sgd[0])
print("RMSE:", train_metrics_sgd[1])
print("MAE:", train_metrics_sgd[2])
print("R2:", train_metrics_sgd[3])
print("Accuracy (%):", train_metrics_sgd[4])

print("\nLinear (SGD) - Testing Results")
print("MSE:", test_metrics_sgd[0])
print("RMSE:", test_metrics_sgd[1])
print("MAE:", test_metrics_sgd[2])
print("R2:", test_metrics_sgd[3])
print("Accuracy (%):", test_metrics_sgd[4])

Linear (SGD) - Training Results
MSE: 525.2319109841978
RMSE: 22.917938628598293
MAE: 10.046347085090042
R2: 0.6702537337236087
Accuracy (%): -14.762923719596643

Linear (SGD) - Testing Results
MSE: 169.72961276425616
RMSE: 13.028031807002014
MAE: 7.480154641167087
R2: 0.7165044820592524
Accuracy (%): 39.09085997501518


# CNN

In [ ]:
scaler_cnn = StandardScaler()
X_train_cnn = scaler_cnn.fit_transform(X_train)
X_test_cnn = scaler_cnn.transform(X_test)

X_train_cnn = X_train_cnn.reshape((X_train_cnn.shape[0], X_train_cnn.shape[1], 1))
X_test_cnn = X_test_cnn.reshape((X_test_cnn.shape[0], X_test_cnn.shape[1], 1))

cnn_model = Sequential([
    Input(shape=(X_train_cnn.shape[1], 1)),
    Conv1D(filters=32, kernel_size=2, activation='relu'),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1)
])

cnn_model.compile(optimizer='adam', loss='mse')

cnn_model.fit(
    X_train_cnn,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=0
)

y_train_pred_cnn = cnn_model.predict(X_train_cnn).flatten()
y_test_pred_cnn = cnn_model.predict(X_test_cnn).flatten()

train_metrics_cnn = evaluate_model(y_train, y_train_pred_cnn)
test_metrics_cnn = evaluate_model(y_test, y_test_pred_cnn)

print("CNN - Training Results")
print("MSE:", train_metrics_cnn[0])
print("RMSE:", train_metrics_cnn[1])
print("MAE:", train_metrics_cnn[2])
print("R2:", train_metrics_cnn[3])
print("Accuracy (%):", train_metrics_cnn[4])

print("\nCNN - Testing Results")
print("MSE:", test_metrics_cnn[0])
print("RMSE:", test_metrics_cnn[1])
print("MAE:", test_metrics_cnn[2])
print("R2:", test_metrics_cnn[3])
print("Accuracy (%):", test_metrics_cnn[4])

3251/3251 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
813/813 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
CNN - Training Results
MSE: 133.93229005759682
RMSE: 11.572911909178123
MAE: 5.468798049092882
R2: 0.9159158618188606
Accuracy (%): 44.31423003552869

CNN - Testing Results
MSE: 85.39756780550869
RMSE: 9.241080445787098
MAE: 5.551878588506118
R2: 0.8573623817222236
Accuracy (%): 53.92089106974303


# **Random Forest**

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

train_metrics_rf = evaluate_model(y_train, y_train_pred_rf)
test_metrics_rf = evaluate_model(y_test, y_test_pred_rf)

print("Random Forest - Training Results")
print("MSE:", train_metrics_rf[0])
print("RMSE:", train_metrics_rf[1])
print("MAE:", train_metrics_rf[2])
print("R2:", train_metrics_rf[3])
print("Accuracy (%):", train_metrics_rf[4])

print("\nRandom Forest - Testing Results")
print("MSE:", test_metrics_rf[0])
print("RMSE:", test_metrics_rf[1])
print("MAE:", test_metrics_rf[2])
print("R2:", test_metrics_rf[3])
print("Accuracy (%):", test_metrics_rf[4])

Random Forest - Training Results
MSE: 6.74553534625936
RMSE: 2.597216846214301
MAE: 1.164703545124132
R2: 0.9957650800571191
Accuracy (%): 94.57193384194395

Random Forest - Testing Results
MSE: 52.462456544860274
RMSE: 7.243097165222919
MAE: 3.3430925322103
R2: 0.9123731501627438
Accuracy (%): 85.51587537952456


# XG Boost

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_train_pred_xgb = xgb_model.predict(X_train)
y_test_pred_xgb = xgb_model.predict(X_test)

train_metrics_xgb = evaluate_model(y_train, y_train_pred_xgb)
test_metrics_xgb = evaluate_model(y_test, y_test_pred_xgb)

print("XGBoost - Training Results")
print("MSE:", train_metrics_xgb[0])
print("RMSE:", train_metrics_xgb[1])
print("MAE:", train_metrics_xgb[2])
print("R2:", train_metrics_xgb[3])
print("Accuracy (%):", train_metrics_xgb[4])

print("\nXGBoost - Testing Results")
print("MSE:", test_metrics_xgb[0])
print("RMSE:", test_metrics_xgb[1])
print("MAE:", test_metrics_xgb[2])
print("R2:", test_metrics_xgb[3])
print("Accuracy (%):", test_metrics_xgb[4])

XGBoost - Training Results
MSE: 40.05484315633713
RMSE: 6.328889567399413
MAE: 3.259694104304128
R2: 0.9748531368698845
Accuracy (%): 80.82589201621101

XGBoost - Testing Results
MSE: 82.41895565127842
RMSE: 9.078488621531584
MAE: 3.385049736010607
R2: 0.8623374899644191
Accuracy (%): 84.8858597534234


# **Light GBM**

In [ ]:
lgbm_model = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

lgbm_model.fit(X_train, y_train)

y_train_pred_lgbm = lgbm_model.predict(X_train)
y_test_pred_lgbm = lgbm_model.predict(X_test)

train_metrics_lgbm = evaluate_model(y_train, y_train_pred_lgbm)
test_metrics_lgbm = evaluate_model(y_test, y_test_pred_lgbm)

print("LightGBM - Training Results")
print("MSE:", train_metrics_lgbm[0])
print("RMSE:", train_metrics_lgbm[1])
print("MAE:", train_metrics_lgbm[2])
print("R2:", train_metrics_lgbm[3])
print("Accuracy (%):", train_metrics_lgbm[4])

print("\nLightGBM - Testing Results")
print("MSE:", test_metrics_lgbm[0])
print("RMSE:", test_metrics_lgbm[1])
print("MAE:", test_metrics_lgbm[2])
print("R2:", test_metrics_lgbm[3])
print("Accuracy (%):", test_metrics_lgbm[4])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008036 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2921
[LightGBM] [Info] Number of data points in the train set: 104002, number of used features: 15
[LightGBM] [Info] Start training from score 25.543202
LightGBM - Training Results
MSE: 48.08808137758477
RMSE: 6.934557042636881
MAE: 3.433440533006722
R2: 0.9698097831547579
Accuracy (%): 78.30679939854627

LightGBM - Testing Results
MSE: 91.49938096353864
RMSE: 9.565530877245582
MAE: 3.4963230537722767
R2: 0.8471706617657536
Accuracy (%): 83.74411562739502


In [ ]:
# =========================
# Final Result Tables Only
# =========================

training_results_table = pd.DataFrame([
    {
        "Model": "Linear (SGD)",
        "MSE": train_metrics_sgd[0],
        "RMSE": train_metrics_sgd[1],
        "MAE": train_metrics_sgd[2],
        "R2": train_metrics_sgd[3],
        "Accuracy (%)": train_metrics_sgd[4]
    },
    {
        "Model": "CNN",
        "MSE": train_metrics_cnn[0],
        "RMSE": train_metrics_cnn[1],
        "MAE": train_metrics_cnn[2],
        "R2": train_metrics_cnn[3],
        "Accuracy (%)": train_metrics_cnn[4]
    },
    {
        "Model": "Random Forest",
        "MSE": train_metrics_rf[0],
        "RMSE": train_metrics_rf[1],
        "MAE": train_metrics_rf[2],
        "R2": train_metrics_rf[3],
        "Accuracy (%)": train_metrics_rf[4]
    },
    {
        "Model": "XGBoost",
        "MSE": train_metrics_xgb[0],
        "RMSE": train_metrics_xgb[1],
        "MAE": train_metrics_xgb[2],
        "R2": train_metrics_xgb[3],
        "Accuracy (%)": train_metrics_xgb[4]
    },
    {
        "Model": "LightGBM",
        "MSE": train_metrics_lgbm[0],
        "RMSE": train_metrics_lgbm[1],
        "MAE": train_metrics_lgbm[2],
        "R2": train_metrics_lgbm[3],
        "Accuracy (%)": train_metrics_lgbm[4]
    }
]).round(4)

testing_results_table = pd.DataFrame([
    {
        "Model": "Linear (SGD)",
        "MSE": test_metrics_sgd[0],
        "RMSE": test_metrics_sgd[1],
        "MAE": test_metrics_sgd[2],
        "R2": test_metrics_sgd[3],
        "Accuracy (%)": test_metrics_sgd[4]
    },
    {
        "Model": "CNN",
        "MSE": test_metrics_cnn[0],
        "RMSE": test_metrics_cnn[1],
        "MAE": test_metrics_cnn[2],
        "R2": test_metrics_cnn[3],
        "Accuracy (%)": test_metrics_cnn[4]
    },
    {
        "Model": "Random Forest",
        "MSE": test_metrics_rf[0],
        "RMSE": test_metrics_rf[1],
        "MAE": test_metrics_rf[2],
        "R2": test_metrics_rf[3],
        "Accuracy (%)": test_metrics_rf[4]
    },
    {
        "Model": "XGBoost",
        "MSE": test_metrics_xgb[0],
        "RMSE": test_metrics_xgb[1],
        "MAE": test_metrics_xgb[2],
        "R2": test_metrics_xgb[3],
        "Accuracy (%)": test_metrics_xgb[4]
    },
    {
        "Model": "LightGBM",
        "MSE": test_metrics_lgbm[0],
        "RMSE": test_metrics_lgbm[1],
        "MAE": test_metrics_lgbm[2],
        "R2": test_metrics_lgbm[3],
        "Accuracy (%)": test_metrics_lgbm[4]
    }
]).round(4)

print("TRAINING RESULTS TABLE")
print(training_results_table)

print("\nTESTING RESULTS TABLE")
print(testing_results_table)

TRAINING RESULTS TABLE
           Model       MSE     RMSE      MAE      R2  Accuracy (%)
0   Linear (SGD)  525.2319  22.9179  10.0463  0.6703      -14.7629
1            CNN  133.9323  11.5729   5.4688  0.9159       44.3142
2  Random Forest    6.7455   2.5972   1.1647  0.9958       94.5719
3        XGBoost   40.0548   6.3289   3.2597  0.9749       80.8259
4       LightGBM   48.0881   6.9346   3.4334  0.9698       78.3068

TESTING RESULTS TABLE
           Model       MSE     RMSE     MAE      R2  Accuracy (%)
0   Linear (SGD)  169.7296  13.0280  7.4802  0.7165       39.0909
1            CNN   85.3976   9.2411  5.5519  0.8574       53.9209
2  Random Forest   52.4625   7.2431  3.3431  0.9124       85.5159
3        XGBoost   82.4190   9.0785  3.3850  0.8623       84.8859
4       LightGBM   91.4994   9.5655  3.4963  0.8472       83.7441


In [ ]:
display(training_results_table)
display(testing_results_table)

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Linear (SGD),525.2319,22.9179,10.0463,0.6703,-14.7629
1,CNN,133.9323,11.5729,5.4688,0.9159,44.3142
2,Random Forest,6.7455,2.5972,1.1647,0.9958,94.5719
3,XGBoost,40.0548,6.3289,3.2597,0.9749,80.8259
4,LightGBM,48.0881,6.9346,3.4334,0.9698,78.3068


,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Linear (SGD),169.7296,13.0280,7.4802,0.7165,39.0909
1,CNN,85.3976,9.2411,5.5519,0.8574,53.9209
2,Random Forest,52.4625,7.2431,3.3431,0.9124,85.5159
3,XGBoost,82.4190,9.0785,3.3850,0.8623,84.8859
4,LightGBM,91.4994,9.5655,3.4963,0.8472,83.7441


In [ ]:
from google.colab import files

files.download("training_results_table.csv")
files.download("testing_results_table.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>